### JAX Conversion of C Statistical Functions

This section converts the C code provided into JAX. We will define equivalent functions using `jax.numpy` for array operations and leverage `jax.jit` for performance optimization on the main computation function.

In [1]:
import jax
import jax.numpy as jnp
import numpy as np # Used for initial array construction due to sequential dependency

# Enable float64 for better precision, matching C's double
jax.config.update("jax_enable_x64", True)

In [2]:
# Equivalent JAX functions

def compute_avg_jax(values):
    return jnp.mean(values)

def compute_mean_abs_deviation_jax(values):
    avg = compute_avg_jax(values)
    return jnp.mean(jnp.abs(values - avg))

def compute_sq_norm_jax(values):
    return jnp.sum(jnp.square(values))

def compute_std_jax(values):
    # JAX's jnp.std with ddof=0 is equivalent to the C formula sqrt(E[X^2] - (E[X])^2)
    return jnp.std(values, ddof=0)

def abs_to_std_jax(a, s):
    # Handle potential division by zero using jnp.where
    return jnp.where(s != 0, (a * a - s * s) / s, jnp.array(jnp.inf, dtype=s.dtype))

@jax.jit
def compute_qoi_jax(values):
    average = compute_avg_jax(values)
    mabs = compute_mean_abs_deviation_jax(values)
    std = compute_std_jax(values)
    a_to_s = abs_to_std_jax(mabs, std)
    return average, mabs, std, a_to_s

In [3]:
# Replicate the C array initialization logic
n = 200
values_list = [0.0] * n # Initialize with zeros

values_list[n - 1] = 32.01

# Fill values in reverse order, as in the C code
for i in range(n - 1, 0, -1):
    current_val_i = values_list[i]
    # Using numpy functions for scalar operations in the loop for simplicity
    # before converting to JAX array
    values_list[i - 1] = 2.0 * np.sqrt(current_val_i) * i + np.exp(-current_val_i)

# Convert the list to a JAX array with float64 dtype
values_jax = jnp.array(values_list, dtype=jnp.float64)

print(f"First 5 elements of JAX array: {values_jax[:5]}")
print(f"Last 5 elements of JAX array: {values_jax[-5:]}")

First 5 elements of JAX array: [ 11.04482737  30.4970529   58.12938971  93.86183189 137.65692947]
Last 5 elements of JAX array: [9.11011193e+04 5.40101386e+04 1.87913569e+04 2.25177975e+03
 3.20100000e+01]


In [4]:
# Compute and print results using JAX
average_jax, mabs_jax, std_jax, as_jax = compute_qoi_jax(values_jax)

print(f"Avg = {average_jax:.6f}\t Mean abs deviation = {mabs_jax:.6f}\t Standard deviation = {std_jax:.6f}\t Our metric = {as_jax:.6f}")

Avg = 51033.476530	 Mean abs deviation = 38991.833912	 Standard deviation = 45120.316766	 Our metric = -11424.562375
